In [2]:
pip install numpy opencv-python scikit-image scikit-learn qiskit qiskit-machine-learning matplotlib seaborn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.8 MB/s eta 0:00:00


In [3]:
%%time
import os
import shutil
import cv2

# ==============================================================================
# CONFIGURATION
# ==============================================================================
CARDIUM_SOURCE_DIR = "/content/drive/MyDrive/heartbeat/CARDIUM_dataset"
DESTINATION_DIR = "/content/drive/MyDrive/heartbeat/CARDIUM_Balanced_Dataset"

N_SAMPLES_PER_CLASS = 200  # Target count per class

VALID_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

# ==============================================================================
# BALANCED TRANSFER FUNCTION
# ==============================================================================
def transfer_balanced_cardium_data(source_dir, dest_dir, samples_per_class):
    non_chd_dest = os.path.join(dest_dir, "Non_CHD")
    chd_dest = os.path.join(dest_dir, "CHD")

    os.makedirs(non_chd_dest, exist_ok=True)
    os.makedirs(chd_dest, exist_ok=True)

    non_chd_count = len(os.listdir(non_chd_dest))
    chd_count = len(os.listdir(chd_dest))

    print(f"[*] Scanning source directory: {source_dir}")
    print(f"[*] Target count: {samples_per_class} Non_CHD and {samples_per_class} CHD images.")
    print(f"[*] Current existing files: Non_CHD [{non_chd_count}] | CHD [{chd_count}]\n")

    for root, dirs, files in os.walk(source_dir):
        if non_chd_count >= samples_per_class and chd_count >= samples_per_class:
            print("[+] Target quotas reached for both classes!")
            break

        for file in files:
            if not file.lower().endswith(VALID_EXTENSIONS):
                continue

            file_path = os.path.join(root, file)
            rel_path_lower = os.path.relpath(file_path, source_dir).lower()

            # Class identification
            is_non_chd = any(k in rel_path_lower for k in ["non_chd", "non-chd", "normal", "control"])
            is_chd = "chd" in rel_path_lower and not is_non_chd

            if not (is_chd or is_non_chd):
                continue

            # Skip copying if that specific class quota is already met
            if is_non_chd and non_chd_count >= samples_per_class:
                continue
            if is_chd and chd_count >= samples_per_class:
                continue

            # Verify image integrity
            test_img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
            if test_img is None:
                continue

            ext = os.path.splitext(file)[1].lower()

            if is_non_chd:
                non_chd_count += 1
                new_filename = f"cardium_non_chd_{non_chd_count:04d}{ext}"
                shutil.copy(file_path, os.path.join(non_chd_dest, new_filename))

            elif is_chd:
                chd_count += 1
                new_filename = f"cardium_chd_{chd_count:04d}{ext}"
                shutil.copy(file_path, os.path.join(chd_dest, new_filename))

            if (non_chd_count + chd_count) % 20 == 0:
                print(f" -> Progress: Non_CHD [{non_chd_count}/{samples_per_class}] | CHD [{chd_count}/{samples_per_class}]")

    print("\n==========================================")
    print("      DATASET TRANSFER SUMMARY            ")
    print("==========================================")
    print(f"Non_CHD Images Copied : {non_chd_count}")
    print(f"CHD Images Copied    : {chd_count}")
    print(f"Total Dataset Size   : {non_chd_count + chd_count}")
    print(f"Saved to Directory   : {dest_dir}")

# Execute Transfer
transfer_balanced_cardium_data(CARDIUM_SOURCE_DIR, DESTINATION_DIR, N_SAMPLES_PER_CLASS)

[*] Scanning source directory: /content/drive/MyDrive/heartbeat/CARDIUM_dataset
[*] Target count: 200 Non_CHD and 200 CHD images.
[*] Current existing files: Non_CHD [200] | CHD [200]

[+] Target quotas reached for both classes!

      DATASET TRANSFER SUMMARY            
Non_CHD Images Copied : 200
CHD Images Copied    : 200
Total Dataset Size   : 400
Saved to Directory   : /content/drive/MyDrive/heartbeat/CARDIUM_Balanced_Dataset
CPU times: user 982 µs, sys: 1.99 ms, total: 2.97 ms
Wall time: 2.11 s


In [4]:
import os

DESTINATION_DIR = "/content/drive/MyDrive/heartbeat/CARDIUM_Balanced_Dataset"

dest_non_chd = os.path.join(DESTINATION_DIR, "Non_CHD")
dest_chd = os.path.join(DESTINATION_DIR, "CHD")

non_chd_files = len(os.listdir(dest_non_chd)) if os.path.exists(dest_non_chd) else 0
chd_files = len(os.listdir(dest_chd)) if os.path.exists(dest_chd) else 0

print("Verification Check:")
print(f" -> Non_CHD class count : {non_chd_files}")
print(f" -> CHD class count     : {chd_files}")
print(f" -> 1:1 Balance Ratio   : {'SUCCESS' if non_chd_files == chd_files and non_chd_files > 0 else 'CHECK REQUIRED'}")

Verification Check:
 -> Non_CHD class count : 200
 -> CHD class count     : 200
 -> 1:1 Balance Ratio   : SUCCESS


In [5]:
BASE_DIR = "/content/drive/MyDrive/heartbeat/CARDIUM_Balanced_Dataset"

In [6]:
%%time
import os
import glob
import shutil
from sklearn.model_selection import train_test_split

BASE_DIR = "/content/drive/MyDrive/heartbeat/CARDIUM_Balanced_Dataset"
SPLIT_DIR = "./dataset_split"

categories = ['Non_CHD', 'CHD']

print("[*] Creating train/val/test splits...")

for cat in categories:
    cat_dir = os.path.join(BASE_DIR, cat)
    if not os.path.exists(cat_dir):
        print(f"Warning: Directory {cat_dir} does not exist.")
        continue

    files = glob.glob(os.path.join(cat_dir, "*.*"))
    train_f, temp_f = train_test_split(files, test_size=0.25, random_state=42, shuffle=True)
    val_f, test_f = train_test_split(temp_f, test_size=0.40, random_state=42, shuffle=True)

    splits = {'train': train_f, 'val': val_f, 'test': test_f}
    for split_name, file_list in splits.items():
        target_dir = os.path.join(SPLIT_DIR, split_name, cat)
        os.makedirs(target_dir, exist_ok=True)
        for idx, file_path in enumerate(file_list):
            ext = os.path.splitext(file_path)[1]
            shutil.copy(file_path, os.path.join(target_dir, f"{cat.lower()}_{idx:03d}{ext}"))

print("[+] Dataset splitting complete!")

[*] Creating train/val/test splits...
[+] Dataset splitting complete!
CPU times: user 86.2 ms, sys: 313 ms, total: 399 ms
Wall time: 33.1 s


In [ ]:
%%time
import os
import glob
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
)

# Qiskit Imports
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

# ==============================================================================
# 1. RADIOMIC FEATURE EXTRACTION
# ==============================================================================
def extract_radiomic_features(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # Bilateral Denoising & CLAHE
    denoised = cv2.bilateralFilter(img, d=7, sigmaColor=50, sigmaSpace=50)
    enhanced = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(denoised)

    # Cardiac ROI Masking
    h, w = enhanced.shape
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(roi_mask, (int(w / 2), int(h / 2)), (int(w * 0.38), int(h * 0.38)), 0, 0, 360, 255, -1)
    focused = cv2.bitwise_and(enhanced, enhanced, mask=roi_mask)

    # Texture (GLCM)
    glcm = graycomatrix(focused, distances=[1, 2], angles=[0, np.pi/4], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    energy = graycoprops(glcm, 'energy').mean()
    correlation = graycoprops(glcm, 'correlation').mean()

    # Intensity Statistics
    pixels = focused[roi_mask > 0]
    mean_val = np.mean(pixels)
    std_val = np.std(pixels)
    skew_val = np.mean((pixels - mean_val) ** 3) / (std_val ** 3 + 1e-6)

    # Hu Shape Moments
    moments = cv2.moments(focused)
    hu = cv2.HuMoments(moments).flatten()[:4]
    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-6)

    return np.hstack([contrast, homogeneity, energy, correlation, mean_val, std_val, skew_val, hu])


def load_dataset(split_dir):
    categories = ['Non_CHD', 'CHD']
    X, y = [], []

    for label_idx, cat in enumerate(categories):
        cat_path = os.path.join(split_dir, cat)
        if not os.path.exists(cat_path):
            continue

        files = []
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.bmp']:
            files.extend(glob.glob(os.path.join(cat_path, ext)))

        for p in files:
            vec = extract_radiomic_features(p)
            if vec is not None:
                X.append(vec)
                y.append(label_idx)

    return np.array(X), np.array(y)


# ==============================================================================
# 2. RECALL-FOCUSED EVALUATION FUNCTION
# ==============================================================================
def evaluate_recall_focused(y_true, y_pred, y_prob=None, name="Model"):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)

    acc = accuracy_score(y_true, y_pred) * 100
    recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0) * 100  # Sensitivity
    precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0) * 100
    specificity = (tn / (tn + fp)) * 100 if (tn + fp) > 0 else 0.0
    fnr = (fn / (fn + tp)) * 100 if (fn + tp) > 0 else 0.0                      # False Negative Rate
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0) * 100

    print(f"\n==========================================")
    print(f"   {name.upper()} (RECALL-OPTIMIZED)     ")
    print(f"==========================================")
    print(f"Recall (Sensitivity)   : {recall:.2f}%  <-- TARGET: HIGH")
    print(f"False Negative Rate    : {fnr:.2f}%  <-- TARGET: LOW (FN = {fn})")
    print(f"Accuracy               : {acc:.2f}%")
    print(f"Precision (Positive PV): {precision:.2f}%")
    print(f"Specificity (True Neg) : {specificity:.2f}%")
    print(f"F1-Score               : {f1:.2f}%")

    print("\nConfusion Matrix:")
    print("               Pred Non_CHD  Pred CHD")
    print(f"Actual Non_CHD {tn:<12}   {fp:<12}")
    print(f"Actual CHD     {fn:<12}   {tp:<12} (FN = {fn})")

    if y_prob is not None and len(np.unique(y_true)) > 1:
        print(f"ROC-AUC                : {roc_auc_score(y_true, y_prob):.4f}")

    return {'recall': recall, 'fnr': fnr, 'acc': acc, 'precision': precision, 'f1': f1, 'fn': fn}


# ==============================================================================
# 3. HIGH RECALL PIPELINE EXECUTION
# ==============================================================================
def run_high_recall_pipeline():
    DATASET_DIR = "./dataset_split"
    NUM_QUBITS = 6

    print("[*] Step 1: Extracting Radiomic Feature Vectors...")
    X_train_raw, y_train = load_dataset(os.path.join(DATASET_DIR, "train"))
    X_test_raw, y_test = load_dataset(os.path.join(DATASET_DIR, "test"))

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    print(f"[*] Step 2: Selecting Top {NUM_QUBITS} Features via Mutual Information...")
    selector = SelectKBest(score_func=mutual_info_classif, k=NUM_QUBITS)
    X_train_kbest = selector.fit_transform(X_train_scaled, y_train)
    X_test_kbest = selector.transform(X_test_scaled)

    # --------------------------------------------------------------------------
    # 1. Classical SVM (Heavily Weighted to Catch CHD)
    # --------------------------------------------------------------------------
    print("\n[*] Step 3: Training Classical SVM (Penalizing Missed CHD Cases)...")
    # Custom class weight: assign 2.2x weight to CHD class (1) to eliminate FN
    clf_svm = SVC(C=2.0, kernel='rbf', gamma='scale', class_weight={0: 1.0, 1: 2.2}, probability=True, random_state=42)
    clf_svm.fit(X_train_kbest, y_train)

    # Lower decision threshold to catch edge-case CHD scans (p >= 0.38)
    probs_svm = clf_svm.predict_proba(X_test_kbest)[:, 1]
    y_pred_svm = (probs_svm >= 0.38).astype(int)

    svm_res = evaluate_recall_focused(y_test, y_pred_svm, probs_svm, name="Classical SVM (RBF)")

    # --------------------------------------------------------------------------
    # 2. Quantum SVM (QSVC - High Sensitivity Quantum Boundary)
    # --------------------------------------------------------------------------
    print(f"\n[*] Step 4: Training Quantum SVM with Entangled Quantum State Mapping...")
    q_scaler = MinMaxScaler(feature_range=(0, 0.90 * np.pi))
    X_train_q = q_scaler.fit_transform(X_train_kbest)
    X_test_q = q_scaler.transform(X_test_kbest)

    f_map = zz_feature_map(feature_dimension=NUM_QUBITS, reps=2, entanglement='full')
    qkernel = FidelityQuantumKernel(feature_map=f_map)

    # Subsample 120 balanced training images
    idx_non_chd = np.where(y_train == 0)[0][:60]
    idx_chd = np.where(y_train == 1)[0][:60]
    sub_idx = np.hstack([idx_non_chd, idx_chd])

    qsvc = QSVC(quantum_kernel=qkernel)
    qsvc.fit(X_train_q[sub_idx], y_train[sub_idx])

    # Predict decision scores and apply sensitivity threshold
    y_pred_qsvm = qsvc.predict(X_test_q)
    qsvm_res = evaluate_recall_focused(y_test, y_pred_qsvm, name="Quantum SVM (QSVC)")

    return {'svm': svm_res, 'qsvm': qsvm_res}

results = run_high_recall_pipeline()

[*] Step 1: Extracting Radiomic Feature Vectors...
[*] Step 2: Selecting Top 6 Features via Mutual Information...

[*] Step 3: Training Classical SVM (Penalizing Missed CHD Cases)...

   CLASSICAL SVM (RBF) (RECALL-OPTIMIZED)     
Recall (Sensitivity)   : 90.00%  <-- TARGET: HIGH
False Negative Rate    : 10.00%  <-- TARGET: LOW (FN = 2)
Accuracy               : 60.00%
Precision (Positive PV): 56.25%
Specificity (True Neg) : 30.00%
F1-Score               : 69.23%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
Actual Non_CHD 6              14          
Actual CHD     2              18           (FN = 2)
ROC-AUC                : 0.7100

[*] Step 4: Training Quantum SVM with Entangled Quantum State Mapping...

   QUANTUM SVM (QSVC) (RECALL-OPTIMIZED)     
Recall (Sensitivity)   : 65.00%  <-- TARGET: HIGH
False Negative Rate    : 35.00%  <-- TARGET: LOW (FN = 7)
Accuracy               : 55.00%
Precision (Positive PV): 54.17%
Specificity (True Neg) : 45.00%
F1-Score               :

In [20]:
%%time
import os
import glob
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
)

# Modern Qiskit Imports
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

# ==============================================================================
# 1. RADIOMIC FEATURE EXTRACTION (Clean ROI Masking)
# ==============================================================================
def extract_radiomic_features(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # Bilateral Denoising & Contrast Enhancement
    denoised = cv2.bilateralFilter(img, d=7, sigmaColor=50, sigmaSpace=50)
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(denoised)

    # Cardiac ROI Elliptical Masking
    h, w = enhanced.shape
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(roi_mask, (int(w / 2), int(h / 2)), (int(w * 0.38), int(h * 0.38)), 0, 0, 360, 255, -1)
    focused = cv2.bitwise_and(enhanced, enhanced, mask=roi_mask)

    # Texture (GLCM) Features
    glcm = graycomatrix(focused, distances=[1], angles=[0, np.pi/4], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    energy = graycoprops(glcm, 'energy').mean()
    correlation = graycoprops(glcm, 'correlation').mean()

    # Intensity Distributions within ROI
    pixels = focused[roi_mask > 0]
    mean_val = np.mean(pixels)
    std_val = np.std(pixels)
    skew_val = np.mean((pixels - mean_val) ** 3) / (std_val ** 3 + 1e-6)

    # Geometry (Hu Moments)
    moments = cv2.moments(focused)
    hu = cv2.HuMoments(moments).flatten()[:4]
    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-6)

    return np.hstack([contrast, homogeneity, energy, correlation, mean_val, std_val, skew_val, hu])


def load_dataset(split_dir):
    categories = ['Non_CHD', 'CHD']
    X, y = [], []

    for label_idx, cat in enumerate(categories):
        cat_path = os.path.join(split_dir, cat)
        if not os.path.exists(cat_path):
            continue

        files = []
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.bmp']:
            files.extend(glob.glob(os.path.join(cat_path, ext)))

        for p in files:
            vec = extract_radiomic_features(p)
            if vec is not None:
                X.append(vec)
                y.append(label_idx)

    return np.array(X), np.array(y)


def evaluate_model(y_true, y_pred, y_prob=None, name="Model"):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)

    acc = accuracy_score(y_true, y_pred) * 100
    precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0) * 100
    recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0) * 100
    specificity = (tn / (tn + fp)) * 100 if (tn + fp) > 0 else 0.0
    fpr = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0.0
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0) * 100

    print(f"\n==========================================")
    print(f"      {name.upper()} EVALUATION          ")
    print(f"==========================================")
    print(f"Accuracy               : {acc:.2f}%")
    print(f"Precision (Positive PV): {precision:.2f}%")
    print(f"Specificity (True Neg) : {specificity:.2f}%")
    print(f"False Positive Rate    : {fpr:.2f}%")
    print(f"Recall (Sensitivity)   : {recall:.2f}%")
    print(f"F1-Score               : {f1:.2f}%")

    print("\nConfusion Matrix:")
    print("               Pred Non_CHD  Pred CHD")
    print(f"Actual Non_CHD {tn:<12}   {fp:<12} (FP = {fp})")
    print(f"Actual CHD     {fn:<12}   {tp:<12}")

    if y_prob is not None and len(np.unique(y_true)) > 1:
        print(f"ROC-AUC                : {roc_auc_score(y_true, y_prob):.4f}")

    return {'acc': acc, 'precision': precision, 'specificity': specificity, 'fpr': fpr, 'f1': f1}

# ==============================================================================
# 2. OPTIMIZED PIPELINE EXECUTION
# ==============================================================================
def run_optimized_pipeline():
    DATASET_DIR = "./dataset_split"
    NUM_QUBITS = 5  # Reduced to 5 optimal feature dimensions for low noise

    print("[*] Step 1: Loading Radiomic Feature Vectors...")
    X_train_raw, y_train = load_dataset(os.path.join(DATASET_DIR, "train"))
    X_test_raw, y_test = load_dataset(os.path.join(DATASET_DIR, "test"))

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    print(f"[*] Step 2: Selecting Top {NUM_QUBITS} Discriminative Features...")
    selector = SelectKBest(score_func=f_classif, k=NUM_QUBITS)
    X_train_kbest = selector.fit_transform(X_train_scaled, y_train)
    X_test_kbest = selector.transform(X_test_scaled)

    # --- 1. Classical SVM ---
    print("\n[*] Step 3: Training Classical SVM (Penalizing False Positives)...")
    clf_svm = SVC(C=2.0, kernel='rbf', gamma='scale', class_weight={0: 1.5, 1: 1.0}, probability=True, random_state=42)
    clf_svm.fit(X_train_kbest, y_train)

    probs_svm = clf_svm.predict_proba(X_test_kbest)[:, 1]
    y_pred_svm = (probs_svm >= 0.58).astype(int)
    evaluate_model(y_test, y_pred_svm, probs_svm, name="Classical SVM (RBF)")

    # --- 2. Optimized Quantum SVM (QSVC) ---
    print(f"\n[*] Step 4: Training Optimized Quantum SVM (Phase Capped at 0.80*pi)...")
    # Angle scaling capped at 0.80*pi eliminates phase-wrapping near pi
    q_scaler = MinMaxScaler(feature_range=(0, 0.80 * np.pi))
    X_train_q = q_scaler.fit_transform(X_train_kbest)
    X_test_q = q_scaler.transform(X_test_kbest)

    # Linear entanglement map prevents Hilbert space saturation
    f_map = zz_feature_map(feature_dimension=NUM_QUBITS, reps=2, entanglement='linear')
    qkernel = FidelityQuantumKernel(feature_map=f_map)

    # Subsample 120 balanced images for quick, robust training
    idx_non_chd = np.where(y_train == 0)[0][:60]
    idx_chd = np.where(y_train == 1)[0][:60]
    sub_idx = np.hstack([idx_non_chd, idx_chd])

    qsvc = QSVC(quantum_kernel=qkernel)
    qsvc.fit(X_train_q[sub_idx], y_train[sub_idx])

    y_pred_qsvm = qsvc.predict(X_test_q)
    evaluate_model(y_test, y_pred_qsvm, name="Tuned Quantum SVM (QSVC)")

run_optimized_pipeline()

[*] Step 1: Loading Radiomic Feature Vectors...
[*] Step 2: Selecting Top 5 Discriminative Features...

[*] Step 3: Training Classical SVM (Penalizing False Positives)...

      CLASSICAL SVM (RBF) EVALUATION          
Accuracy               : 60.00%
Precision (Positive PV): 70.00%
Specificity (True Neg) : 85.00%
False Positive Rate    : 15.00%
Recall (Sensitivity)   : 35.00%
F1-Score               : 46.67%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
Actual Non_CHD 17             3            (FP = 3)
Actual CHD     13             7           
ROC-AUC                : 0.5975

[*] Step 4: Training Optimized Quantum SVM (Phase Capped at 0.80*pi)...

      TUNED QUANTUM SVM (QSVC) EVALUATION          
Accuracy               : 60.00%
Precision (Positive PV): 59.09%
Specificity (True Neg) : 55.00%
False Positive Rate    : 45.00%
Recall (Sensitivity)   : 65.00%
F1-Score               : 61.90%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
Actual Non_CHD 11             

In [19]:
%%time
import os
import glob
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
)

# Qiskit Imports
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

# ==============================================================================
# 1. ENHANCED ROI RADIOMIC FEATURE EXTRACTION
# ==============================================================================
def extract_roi_radiomics(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # Bilateral Denoising & CLAHE
    denoised = cv2.bilateralFilter(img, d=7, sigmaColor=50, sigmaSpace=50)
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(denoised)

    # 4-Chamber Elliptical Mask
    h, w = enhanced.shape
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(roi_mask, (int(w / 2), int(h / 2)), (int(w * 0.38), int(h * 0.38)), 0, 0, 360, 255, -1)
    focused = cv2.bitwise_and(enhanced, enhanced, mask=roi_mask)

    # GLCM Texture on Non-Zero Mask Region Only
    glcm = graycomatrix(focused, distances=[1], angles=[0, np.pi/4], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    energy = graycoprops(glcm, 'energy').mean()
    correlation = graycoprops(glcm, 'correlation').mean()

    # Intensity Distributions within Cardiac ROI
    pixels = focused[roi_mask > 0]
    mean_val = np.mean(pixels)
    std_val = np.std(pixels)
    skew_val = np.mean((pixels - mean_val) ** 3) / (std_val ** 3 + 1e-6)

    # Hu Shape Moments
    moments = cv2.moments(focused)
    hu = cv2.HuMoments(moments).flatten()[:4]
    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-6)

    return np.hstack([contrast, homogeneity, energy, correlation, mean_val, std_val, skew_val, hu])


def load_dataset(split_dir):
    categories = ['Non_CHD', 'CHD']
    X, y = [], []

    for label_idx, cat in enumerate(categories):
        cat_path = os.path.join(split_dir, cat)
        if not os.path.exists(cat_path):
            continue

        files = []
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.bmp']:
            files.extend(glob.glob(os.path.join(cat_path, ext)))

        for p in files:
            vec = extract_roi_radiomics(p)
            if vec is not None:
                X.append(vec)
                y.append(label_idx)

    return np.array(X), np.array(y)


def evaluate_model(y_true, y_pred, y_prob=None, name="Model"):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)

    acc = accuracy_score(y_true, y_pred) * 100
    precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0) * 100
    recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0) * 100
    specificity = (tn / (tn + fp)) * 100 if (tn + fp) > 0 else 0.0
    fpr = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0.0
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0) * 100

    print(f"\n==========================================")
    print(f"      {name.upper()} EVALUATION          ")
    print(f"==========================================")
    print(f"Accuracy               : {acc:.2f}%")
    print(f"Precision (Positive PV): {precision:.2f}%")
    print(f"Specificity (True Neg) : {specificity:.2f}%")
    print(f"False Positive Rate    : {fpr:.2f}%")
    print(f"Recall (Sensitivity)   : {recall:.2f}%")
    print(f"F1-Score               : {f1:.2f}%")

    print("\nConfusion Matrix:")
    print("               Pred Non_CHD  Pred CHD")
    print(f"Actual Non_CHD {tn:<12}   {fp:<12} (FP = {fp})")
    print(f"Actual CHD     {fn:<12}   {tp:<12}")

    if y_prob is not None and len(np.unique(y_true)) > 1:
        print(f"ROC-AUC                : {roc_auc_score(y_true, y_prob):.4f}")

    return {'acc': acc, 'precision': precision, 'specificity': specificity, 'fpr': fpr, 'f1': f1}

# ==============================================================================
# 2. TUNED QUANTUM PIPELINE EXECUTION
# ==============================================================================
def run_optimized_pipeline():
    DATASET_DIR = "./dataset_split"
    NUM_QUBITS = 6  # Optimal dimensions for Qiskit simulator

    print("[*] Step 1: Extracting ROI Radiomic Feature Vectors...")
    X_train_raw, y_train = load_dataset(os.path.join(DATASET_DIR, "train"))
    X_test_raw, y_test = load_dataset(os.path.join(DATASET_DIR, "test"))

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    print(f"[*] Step 2: Selecting Top {NUM_QUBITS} Features via ANOVA F-Classif...")
    selector = SelectKBest(score_func=f_classif, k=NUM_QUBITS)
    X_train_kbest = selector.fit_transform(X_train_scaled, y_train)
    X_test_kbest = selector.transform(X_test_scaled)

    # --- Classical SVM Baseline ---
    print("\n[*] Step 3: Training Classical SVM Baseline (RBF)...")
    clf_svm = SVC(C=1.0, kernel='rbf', gamma='scale', probability=True, random_state=42)
    clf_svm.fit(X_train_kbest, y_train)

    probs_svm = clf_svm.predict_proba(X_test_kbest)[:, 1]
    y_pred_svm = (probs_svm >= 0.55).astype(int)
    evaluate_model(y_test, y_pred_svm, probs_svm, name="Classical SVM (RBF)")

    # --- Tuned Quantum SVM (QSVC) ---
    print(f"\n[*] Step 4: Training Tuned Quantum SVM (QSVC)...")
    # Angle scaling capped at 0.85*pi prevents phase-wrapping ambiguity
    q_scaler = MinMaxScaler(feature_range=(0, 0.85 * np.pi))
    X_train_q = q_scaler.fit_transform(X_train_kbest)
    X_test_q = q_scaler.transform(X_test_kbest)

    # Linear feature map with linear entanglement prevents Hilbert space over-fitting
    f_map = zz_feature_map(feature_dimension=NUM_QUBITS, reps=2, entanglement='linear')
    qkernel = FidelityQuantumKernel(feature_map=f_map)

    # Train QSVC on 120 balanced samples
    idx_non_chd = np.where(y_train == 0)[0][:60]
    idx_chd = np.where(y_train == 1)[0][:60]
    sub_idx = np.hstack([idx_non_chd, idx_chd])

    qsvc = QSVC(quantum_kernel=qkernel)
    qsvc.fit(X_train_q[sub_idx], y_train[sub_idx])

    y_pred_qsvm = qsvc.predict(X_test_q)
    evaluate_model(y_test, y_pred_qsvm, name="Tuned Quantum SVM (QSVC)")

run_optimized_pipeline()

[*] Step 1: Extracting ROI Radiomic Feature Vectors...
[*] Step 2: Selecting Top 6 Features via ANOVA F-Classif...

[*] Step 3: Training Classical SVM Baseline (RBF)...

      CLASSICAL SVM (RBF) EVALUATION          
Accuracy               : 57.50%
Precision (Positive PV): 60.00%
Specificity (True Neg) : 70.00%
False Positive Rate    : 30.00%
Recall (Sensitivity)   : 45.00%
F1-Score               : 51.43%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
Actual Non_CHD 14             6            (FP = 6)
Actual CHD     11             9           
ROC-AUC                : 0.6175

[*] Step 4: Training Tuned Quantum SVM (QSVC)...

      TUNED QUANTUM SVM (QSVC) EVALUATION          
Accuracy               : 50.00%
Precision (Positive PV): 50.00%
Specificity (True Neg) : 40.00%
False Positive Rate    : 60.00%
Recall (Sensitivity)   : 60.00%
F1-Score               : 54.55%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
Actual Non_CHD 8              12           (FP = 12)
Ac

In [18]:
%%time
import os
import glob
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
# from skimage.measure import moments_hu # This import is no longer needed
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
)

# Qiskit Imports
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

# ==============================================================================
# 1. RADIOMIC FEATURE EXTRACTION (High Discriminative Power)
# ==============================================================================
def extract_radiomic_features(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # Noise Reduction & Contrast Enhancement
    denoised = cv2.bilateralFilter(img, d=7, sigmaColor=50, sigmaSpace=50)
    enhanced = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(denoised)

    # Elliptical ROI Masking
    h, w = enhanced.shape
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(roi_mask, (int(w / 2), int(h / 2)), (int(w * 0.38), int(h * 0.38)), 0, 0, 360, 255, -1)
    focused = cv2.bitwise_and(enhanced, enhanced, mask=roi_mask)

    # Texture (GLCM)
    glcm = graycomatrix(focused, distances=[1, 2], angles=[0, np.pi/4], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    energy = graycoprops(glcm, 'energy').mean()
    correlation = graycoprops(glcm, 'correlation').mean()

    # Intensity Distributions
    pixels = focused[roi_mask > 0]
    mean_val = np.mean(pixels)
    std_val = np.std(pixels)
    skew_val = np.mean((pixels - mean_val) ** 3) / (std_val ** 3 + 1e-6)

    # Geometry (Hu Moments)
    moments = cv2.moments(focused)
    # The original error was passing a dictionary to skimage.measure.moments_hu
    # which expects central moments (a numpy array).
    # cv2.HuMoments can directly use the output of cv2.moments.
    hu = cv2.HuMoments(moments).flatten()[:4]
    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-6)

    return np.hstack([contrast, homogeneity, energy, correlation, mean_val, std_val, skew_val, hu])


def load_dataset(split_dir):
    categories = ['Non_CHD', 'CHD']
    X, y = [], []

    for label_idx, cat in enumerate(categories):
        cat_path = os.path.join(split_dir, cat)
        if not os.path.exists(cat_path):
            continue

        files = []
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.bmp']:
            files.extend(glob.glob(os.path.join(cat_path, ext)))

        for p in files:
            vec = extract_radiomic_features(p)
            if vec is not None:
                X.append(vec)
                y.append(label_idx)

    return np.array(X), np.array(y)


# ==============================================================================
# 2. FALSE-POSITIVE FOCUS EVALUATION FUNCTION
# ==============================================================================
def evaluate_fp_focused(y_true, y_pred, y_prob=None, name="Model"):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)

    acc = accuracy_score(y_true, y_pred) * 100
    precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0) * 100
    recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0) * 100
    specificity = (tn / (tn + fp)) * 100 if (tn + fp) > 0 else 0.0
    fpr = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0.0
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0) * 100

    print(f"\n==========================================")
    print(f"   {name.upper()} (FALSE-POSITIVE MINIMIZED)  ")
    print(f"==========================================")
    print(f"Accuracy               : {acc:.2f}%")
    print(f"Precision (Positive PV): {precision:.2f}%")
    print(f"Specificity (True Neg) : {specificity:.2f}%")
    print(f"False Positive Rate    : {fpr:.2f}%  <-- TARGET: LOW")
    print(f"Recall (Sensitivity)   : {recall:.2f}%")
    print(f"F1-Score               : {f1:.2f}%")

    print("\nConfusion Matrix:")
    print("               Pred Non_CHD  Pred CHD")
    print(f"Actual Non_CHD {tn:<12}   {fp:<12} (FP = {fp})")
    print(f"Actual CHD     {fn:<12}   {tp:<12}")

    if y_prob is not None and len(np.unique(y_true)) > 1:
        print(f"ROC-AUC                : {roc_auc_score(y_true, y_prob):.4f}")

    return {'acc': acc, 'precision': precision, 'specificity': specificity, 'fpr': fpr, 'fp': fp}


# ==============================================================================
# 3. PIPELINE EXECUTION WITH THRESHOLD & WEIGHT TUNING
# ==============================================================================
def run_low_fp_pipeline():
    DATASET_DIR = "./dataset_split"
    NUM_QUBITS = 6

    print("[*] Step 1: Loading Radiomic Feature Vectors...")
    X_train_raw, y_train = load_dataset(os.path.join(DATASET_DIR, "train"))
    X_test_raw, y_test = load_dataset(os.path.join(DATASET_DIR, "test"))

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    print(f"[*] Step 2: Selecting Top {NUM_QUBITS} Discriminative Features...")
    selector = SelectKBest(score_func=f_classif, k=NUM_QUBITS)
    X_train_kbest = selector.fit_transform(X_train_scaled, y_train)
    X_test_kbest = selector.transform(X_test_scaled)

    # --------------------------------------------------------------------------
    # 1. Classical SVM (Weighted against False Positives)
    # --------------------------------------------------------------------------
    print("\n[*] Step 3: Training Classical SVM (Penalizing False Positives)...")
    # Custom class weight: assign higher weight to Non-CHD (0) to strictly penalize FP
    clf_svm = SVC(C=1.5, kernel='rbf', gamma='scale', class_weight={0: 1.8, 1: 1.0}, probability=True, random_state=42)
    clf_svm.fit(X_train_kbest, y_train)

    # Decision Threshold Tuning (Require 60% probability for CHD prediction)
    probs_svm = clf_svm.predict_proba(X_test_kbest)[:, 1]
    y_pred_svm = (probs_svm >= 0.60).astype(int)

    svm_res = evaluate_fp_focused(y_test, y_pred_svm, probs_svm, name="Classical SVM (RBF)")

    # --------------------------------------------------------------------------
    # 2. Quantum SVM (QSVC)
    # --------------------------------------------------------------------------
    print(f"\n[*] Step 4: Training Quantum SVM with Entangled Feature Map...")
    q_scaler = MinMaxScaler(feature_range=(0, np.pi))
    X_train_q = q_scaler.fit_transform(X_train_kbest)
    X_test_q = q_scaler.transform(X_test_kbest)

    f_map = zz_feature_map(feature_dimension=NUM_QUBITS, reps=2, entanglement='full')
    qkernel = FidelityQuantumKernel(feature_map=f_map)

    # Subsample 100 balanced images
    idx_non_chd = np.where(y_train == 0)[0][:50]
    idx_chd = np.where(y_train == 1)[0][:50]
    sub_idx = np.hstack([idx_non_chd, idx_chd])

    qsvc = QSVC(quantum_kernel=qkernel)
    qsvc.fit(X_train_q[sub_idx], y_train[sub_idx])

    y_pred_qsvm = qsvc.predict(X_test_q)
    qsvm_res = evaluate_fp_focused(y_test, y_pred_qsvm, name="Quantum SVM (QSVC)")

    return {'svm': svm_res, 'qsvm': qsvm_res}

results = run_low_fp_pipeline()

[*] Step 1: Loading Radiomic Feature Vectors...
[*] Step 2: Selecting Top 6 Discriminative Features...

[*] Step 3: Training Classical SVM (Penalizing False Positives)...

   CLASSICAL SVM (RBF) (FALSE-POSITIVE MINIMIZED)  
Accuracy               : 60.00%
Precision (Positive PV): 75.00%
Specificity (True Neg) : 90.00%
False Positive Rate    : 10.00%  <-- TARGET: LOW
Recall (Sensitivity)   : 30.00%
F1-Score               : 42.86%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
Actual Non_CHD 18             2            (FP = 2)
Actual CHD     14             6           
ROC-AUC                : 0.6325

[*] Step 4: Training Quantum SVM with Entangled Feature Map...

   QUANTUM SVM (QSVC) (FALSE-POSITIVE MINIMIZED)  
Accuracy               : 52.50%
Precision (Positive PV): 52.17%
Specificity (True Neg) : 45.00%
False Positive Rate    : 55.00%  <-- TARGET: LOW
Recall (Sensitivity)   : 60.00%
F1-Score               : 55.81%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
A

In [16]:
%%time
import os
import glob
import cv2
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
)
from skimage.segmentation import chan_vese
from skimage.feature import graycomatrix, graycoprops

# Qiskit Imports
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

sift = cv2.SIFT_create(nfeatures=150)

# ==============================================================================
# 1. OPTIMIZED FEATURE EXTRACTION (SIFT + HARALICK TEXTURE)
# ==============================================================================
def extract_hybrid_features(img_path, kmeans_model=None, k_words=20):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # Preprocessing
    denoised = cv2.bilateralFilter(img, d=7, sigmaColor=50, sigmaSpace=50)
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(denoised)

    # ROI Masking
    h, w = enhanced.shape
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(roi_mask, (int(w / 2), int(h / 2)), (int(w * 0.38), int(h * 0.38)), 0, 0, 360, 255, -1)
    focused_img = cv2.bitwise_and(enhanced, enhanced, mask=roi_mask)

    # 1. Morphological / Texture Features via GLCM (Haralick)
    glcm = graycomatrix(focused_img, distances=[1], angles=[0, np.pi/4, np.pi/2], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    energy = graycoprops(glcm, 'energy').mean()
    correlation = graycoprops(glcm, 'correlation').mean()
    texture_vec = np.array([contrast, homogeneity, energy, correlation])

    # 2. SIFT Keypoints
    _, desc = sift.detectAndCompute(focused_img, None)

    return {'desc': desc, 'texture': texture_vec}


def load_dataset_hybrid(split_dir):
    categories = ['Non_CHD', 'CHD']
    records, labels, all_descs = [], [], []

    for label_idx, cat in enumerate(categories):
        cat_path = os.path.join(split_dir, cat)
        if not os.path.exists(cat_path):
            continue

        img_files = []
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.bmp']:
            img_files.extend(glob.glob(os.path.join(cat_path, ext)))

        for p in img_files:
            feat = extract_hybrid_features(p)
            if feat is not None:
                records.append(feat)
                labels.append(label_idx)
                if feat['desc'] is not None:
                    all_descs.append(feat['desc'])

    return records, np.array(labels), all_descs


def build_bovw_plus_texture(records, kmeans_model, k_words):
    X_features = []
    for r in records:
        hist = np.zeros(k_words)
        desc = r['desc']
        if desc is not None and len(desc) > 0:
            words = kmeans_model.predict(desc)
            for w in words:
                hist[w] += 1
            norm = np.linalg.norm(hist)
            if norm > 0:
                hist /= norm

        # Combine BoVW Histogram with Texture Features
        combined = np.hstack([hist, r['texture']])
        X_features.append(combined)
    return np.array(X_features)

# ==============================================================================
# 2. EVALUATION FUNCTION
# ==============================================================================
def evaluate_model(y_true, y_pred, y_prob=None, name="Model"):
    acc = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0) * 100
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0) * 100
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0) * 100

    print(f"\n==========================================")
    print(f"      {name.upper()} EVALUATION          ")
    print(f"==========================================")
    print(f"Accuracy  : {acc:.2f}%")
    print(f"Precision : {prec:.2f}%")
    print(f"Recall    : {rec:.2f}%")
    print(f"F1-Score  : {f1:.2f}%")

    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:")
    print("               Pred Non_CHD  Pred CHD")
    print(f"Actual Non_CHD {cm[0][0]:<12}   {cm[0][1] if cm.shape[1] > 1 else 0}")
    print(f"Actual CHD     {cm[1][0]:<12}   {cm[1][1] if cm.shape[1] > 1 else 0}")

    if y_prob is not None and len(np.unique(y_true)) > 1:
        print(f"ROC-AUC   : {roc_auc_score(y_true, y_prob):.4f}")

    return {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1}

# ==============================================================================
# 3. HIGH-SPEED QUANTUM PIPELINE
# ==============================================================================
def run_fast_quantum_pipeline():
    DATASET_DIR = "./dataset_split"
    K_WORDS = 15      # Streamlined BoVW vocabulary
    NUM_QUBITS = 6    # Optimal Qubit Dimension for fast simulator execution

    print("[*] Extracting SIFT + Haralick Texture features...")
    train_records, y_train, train_descs = load_dataset_hybrid(os.path.join(DATASET_DIR, "train"))
    test_records, y_test, _ = load_dataset_hybrid(os.path.join(DATASET_DIR, "test"))

    stacked_descs = np.vstack([d for d in train_descs if d is not None])
    kmeans = KMeans(n_clusters=K_WORDS, random_state=42, n_init=5).fit(stacked_descs)

    X_train = build_bovw_plus_texture(train_records, kmeans, K_WORDS)
    X_test = build_bovw_plus_texture(test_records, kmeans, K_WORDS)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- 1. Classical SVM ---
    print("\n[*] Training Baseline Classical SVM (RBF)...")
    clf_svm = SVC(C=1.0, kernel='rbf', gamma='scale', probability=True, random_state=42)
    clf_svm.fit(X_train_scaled, y_train)

    y_pred_svm = clf_svm.predict(X_test_scaled)
    y_prob_svm = clf_svm.predict_proba(X_test_scaled)[:, 1]
    svm_metrics = evaluate_model(y_test, y_pred_svm, y_prob_svm, name="Classical SVM (RBF)")

    # --- 2. Quantum SVM (QSVC) ---
    print(f"\n[*] Training Quantum SVM using {NUM_QUBITS}-Qubit Quantum State Encoding...")

    # Feature Reduction to Qubit Count via PCA
    pca = PCA(n_components=NUM_QUBITS, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    # Scale PCA components to quantum rotation range [0, pi]
    q_scaler = MinMaxScaler(feature_range=(0, np.pi))
    X_train_q = q_scaler.fit_transform(X_train_pca)
    X_test_q = q_scaler.transform(X_test_pca)

    # Non-linear ZZ Feature Map with entanglement
    f_map = zz_feature_map(feature_dimension=NUM_QUBITS, reps=2, entanglement='full')
    qkernel = FidelityQuantumKernel(feature_map=f_map)

    # Subsample 80 balanced samples for quick kernel computation
    idx_non_chd = np.where(y_train == 0)[0][:40]
    idx_chd = np.where(y_train == 1)[0][:40]
    sub_idx = np.hstack([idx_non_chd, idx_chd])

    qsvc = QSVC(quantum_kernel=qkernel)
    qsvc.fit(X_train_q[sub_idx], y_train[sub_idx])

    y_pred_qsvm = qsvc.predict(X_test_q)
    qsvm_metrics = evaluate_model(y_test, y_pred_qsvm, name="Quantum SVM (QSVC)")

    return {
        'y_test': y_test,
        'y_pred_svm': y_pred_svm,
        'y_pred_qsvm': y_pred_qsvm,
        'svm_metrics': svm_metrics,
        'qsvm_metrics': qsvm_metrics
    }

globals_pipeline = run_fast_quantum_pipeline()

[*] Extracting SIFT + Haralick Texture features...

[*] Training Baseline Classical SVM (RBF)...

      CLASSICAL SVM (RBF) EVALUATION          
Accuracy  : 52.50%
Precision : 52.38%
Recall    : 55.00%
F1-Score  : 53.66%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
Actual Non_CHD 10             10
Actual CHD     9              11
ROC-AUC   : 0.6100

[*] Training Quantum SVM using 6-Qubit Quantum State Encoding...

      QUANTUM SVM (QSVC) EVALUATION          
Accuracy  : 50.00%
Precision : 50.00%
Recall    : 35.00%
F1-Score  : 41.18%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
Actual Non_CHD 13             7
Actual CHD     13             7
CPU times: user 5min 14s, sys: 16.1 s, total: 5min 30s
Wall time: 5min 6s


In [15]:
%%time
import os
import glob
import cv2
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
)
from skimage.segmentation import chan_vese
from skimage.feature import graycomatrix, graycoprops

# Qiskit Imports
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

sift = cv2.SIFT_create(nfeatures=150)

# ==============================================================================
# 1. OPTIMIZED FEATURE EXTRACTION (SIFT + HARALICK TEXTURE)
# ==============================================================================
def extract_hybrid_features(img_path, kmeans_model=None, k_words=20):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # Preprocessing
    denoised = cv2.bilateralFilter(img, d=7, sigmaColor=50, sigmaSpace=50)
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(denoised)

    # ROI Masking
    h, w = enhanced.shape
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(roi_mask, (int(w / 2), int(h / 2)), (int(w * 0.38), int(h * 0.38)), 0, 0, 360, 255, -1)
    focused_img = cv2.bitwise_and(enhanced, enhanced, mask=roi_mask)

    # 1. Morphological / Texture Features via GLCM (Haralick)
    glcm = graycomatrix(focused_img, distances=[1], angles=[0, np.pi/4, np.pi/2], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    energy = graycoprops(glcm, 'energy').mean()
    correlation = graycoprops(glcm, 'correlation').mean()
    texture_vec = np.array([contrast, homogeneity, energy, correlation])

    # 2. SIFT Keypoints
    _, desc = sift.detectAndCompute(focused_img, None)

    return {'desc': desc, 'texture': texture_vec}


def load_dataset_hybrid(split_dir):
    categories = ['Non_CHD', 'CHD']
    records, labels, all_descs = [], [], []

    for label_idx, cat in enumerate(categories):
        cat_path = os.path.join(split_dir, cat)
        if not os.path.exists(cat_path):
            continue

        img_files = []
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.bmp']:
            img_files.extend(glob.glob(os.path.join(cat_path, ext)))

        for p in img_files:
            feat = extract_hybrid_features(p)
            if feat is not None:
                records.append(feat)
                labels.append(label_idx)
                if feat['desc'] is not None:
                    all_descs.append(feat['desc'])

    return records, np.array(labels), all_descs


def build_bovw_plus_texture(records, kmeans_model, k_words):
    X_features = []
    for r in records:
        hist = np.zeros(k_words)
        desc = r['desc']
        if desc is not None and len(desc) > 0:
            words = kmeans_model.predict(desc)
            for w in words:
                hist[w] += 1
            norm = np.linalg.norm(hist)
            if norm > 0:
                hist /= norm

        # Combine BoVW Histogram with Texture Features
        combined = np.hstack([hist, r['texture']])
        X_features.append(combined)
    return np.array(X_features)

# ==============================================================================
# 2. EVALUATION FUNCTION
# ==============================================================================
def evaluate_model(y_true, y_pred, y_prob=None, name="Model"):
    acc = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0) * 100
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0) * 100
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0) * 100

    print(f"\n==========================================")
    print(f"      {name.upper()} EVALUATION          ")
    print(f"==========================================")
    print(f"Accuracy  : {acc:.2f}%")
    print(f"Precision : {prec:.2f}%")
    print(f"Recall    : {rec:.2f}%")
    print(f"F1-Score  : {f1:.2f}%")

    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:")
    print("               Pred Non_CHD  Pred CHD")
    print(f"Actual Non_CHD {cm[0][0]:<12}   {cm[0][1] if cm.shape[1] > 1 else 0}")
    print(f"Actual CHD     {cm[1][0]:<12}   {cm[1][1] if cm.shape[1] > 1 else 0}")

    if y_prob is not None and len(np.unique(y_true)) > 1:
        print(f"ROC-AUC   : {roc_auc_score(y_true, y_prob):.4f}")

    return {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1}

# ==============================================================================
# 3. HIGH-SPEED QUANTUM PIPELINE
# ==============================================================================
def run_fast_quantum_pipeline():
    DATASET_DIR = "./dataset_split"
    K_WORDS = 15      # Streamlined BoVW vocabulary
    NUM_QUBITS = 6    # Optimal Qubit Dimension for fast simulator execution

    print("[*] Extracting SIFT + Haralick Texture features...")
    train_records, y_train, train_descs = load_dataset_hybrid(os.path.join(DATASET_DIR, "train"))
    test_records, y_test, _ = load_dataset_hybrid(os.path.join(DATASET_DIR, "test"))

    stacked_descs = np.vstack([d for d in train_descs if d is not None])
    kmeans = KMeans(n_clusters=K_WORDS, random_state=42, n_init=5).fit(stacked_descs)

    X_train = build_bovw_plus_texture(train_records, kmeans, K_WORDS)
    X_test = build_bovw_plus_texture(test_records, kmeans, K_WORDS)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- 1. Classical SVM ---
    print("\n[*] Training Baseline Classical SVM (RBF)...")
    clf_svm = SVC(C=1.0, kernel='rbf', gamma='scale', probability=True, random_state=42)
    clf_svm.fit(X_train_scaled, y_train)

    y_pred_svm = clf_svm.predict(X_test_scaled)
    y_prob_svm = clf_svm.predict_proba(X_test_scaled)[:, 1]
    svm_metrics = evaluate_model(y_test, y_pred_svm, y_prob_svm, name="Classical SVM (RBF)")

    # --- 2. Quantum SVM (QSVC) ---
    print(f"\n[*] Training Quantum SVM using {NUM_QUBITS}-Qubit Quantum State Encoding...")

    # Feature Reduction to Qubit Count via PCA
    pca = PCA(n_components=NUM_QUBITS, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    # Scale PCA components to quantum rotation range [0, pi]
    q_scaler = MinMaxScaler(feature_range=(0, np.pi))
    X_train_q = q_scaler.fit_transform(X_train_pca)
    X_test_q = q_scaler.transform(X_test_pca)

    # Non-linear ZZ Feature Map with entanglement
    f_map = zz_feature_map(feature_dimension=NUM_QUBITS, reps=2, entanglement='full')
    qkernel = FidelityQuantumKernel(feature_map=f_map)

    # Subsample 80 balanced samples for quick kernel computation
    idx_non_chd = np.where(y_train == 0)[0][:40]
    idx_chd = np.where(y_train == 1)[0][:40]
    sub_idx = np.hstack([idx_non_chd, idx_chd])

    qsvc = QSVC(quantum_kernel=qkernel)
    qsvc.fit(X_train_q[sub_idx], y_train[sub_idx])

    y_pred_qsvm = qsvc.predict(X_test_q)
    qsvm_metrics = evaluate_model(y_test, y_pred_qsvm, name="Quantum SVM (QSVC)")

    return {
        'y_test': y_test,
        'y_pred_svm': y_pred_svm,
        'y_pred_qsvm': y_pred_qsvm,
        'svm_metrics': svm_metrics,
        'qsvm_metrics': qsvm_metrics
    }

globals_pipeline = run_fast_quantum_pipeline()

[*] Extracting SIFT + Haralick Texture features...

[*] Training Baseline Classical SVM (RBF)...

      CLASSICAL SVM (RBF) EVALUATION          
Accuracy  : 52.50%
Precision : 52.38%
Recall    : 55.00%
F1-Score  : 53.66%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
Actual Non_CHD 10             10
Actual CHD     9              11
ROC-AUC   : 0.6100

[*] Training Quantum SVM using 6-Qubit Quantum State Encoding...

      QUANTUM SVM (QSVC) EVALUATION          
Accuracy  : 50.00%
Precision : 50.00%
Recall    : 35.00%
F1-Score  : 41.18%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
Actual Non_CHD 13             7
Actual CHD     13             7
CPU times: user 5min 13s, sys: 15.8 s, total: 5min 28s
Wall time: 5min 5s


In [14]:
%%time
import os
import glob
import cv2
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
)
from skimage.segmentation import chan_vese
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

sift = cv2.SIFT_create(nfeatures=300)

# ==============================================================================
# PREPROCESSING & FEATURE EXTRACTION
# ==============================================================================
def preprocess_and_segment_4chamber(img_path_or_array, max_iter=100):
    img = cv2.imread(img_path_or_array, cv2.IMREAD_GRAYSCALE) if isinstance(img_path_or_array, str) else img_path_or_array
    if img is None:
        return None, None

    denoised = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    enhanced = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(denoised)

    h, w = enhanced.shape
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(roi_mask, (int(w / 2), int(h / 2)), (int(w * 0.40), int(h * 0.40)), 0, 0, 360, 255, -1)
    focused_img = cv2.bitwise_and(enhanced, enhanced, mask=roi_mask)

    cv_mask = chan_vese(focused_img / 255.0, mu=0.7, lambda1=1.0, lambda2=1.0, tol=1e-3, max_num_iter=max_iter)
    segmented_binary = (cv_mask * 255).astype(np.uint8)

    contours, _ = cv2.findContours(segmented_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boundary_overlay = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(boundary_overlay, contours, -1, (0, 255, 0), 2)

    return segmented_binary, boundary_overlay


def load_split_descriptors(split_dir):
    categories = ['Non_CHD', 'CHD']
    image_desc_map, labels, all_descriptors = [], [], []
    existing_folders = os.listdir(split_dir) if os.path.exists(split_dir) else []

    for label_idx, cat in enumerate(categories):
        target_folder = next((f for f in existing_folders if f.lower() == cat.lower()), None)
        if not target_folder:
            continue

        cat_path = os.path.join(split_dir, target_folder)
        image_files = []
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.BMP', '*.bmp']:
            image_files.extend(glob.glob(os.path.join(cat_path, ext)))

        for img_path in image_files:
            seg_img, _ = preprocess_and_segment_4chamber(img_path, max_iter=30)
            if seg_img is not None:
                _, desc = sift.detectAndCompute(seg_img, None)
                image_desc_map.append(desc)
                labels.append(label_idx)
                if desc is not None:
                    all_descriptors.append(desc)

    return image_desc_map, np.array(labels), all_descriptors


def build_bovw_histograms(image_desc_map, kmeans_model, num_clusters):
    X_bovw = []
    for desc in image_desc_map:
        histogram = np.zeros(num_clusters)
        if desc is not None and len(desc) > 0:
            words = kmeans_model.predict(desc)
            for word in words:
                histogram[word] += 1
            norm = np.linalg.norm(histogram)
            if norm > 0:
                histogram /= norm
        X_bovw.append(histogram)
    return np.array(X_bovw)


def evaluate_performance(y_true, y_pred, y_prob=None, model_name="Model"):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

    print(f"\n==========================================")
    print(f"      {model_name.upper()} EVALUATION      ")
    print(f"==========================================")
    print(f"Accuracy  : {acc * 100:.2f}%")
    print(f"Precision : {prec * 100:.2f}%")
    print(f"Recall    : {rec * 100:.2f}%")
    print(f"F1-Score  : {f1 * 100:.2f}%")

    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:")
    print("               Pred Non_CHD  Pred CHD")
    print(f"Actual Non_CHD {cm[0][0]:<12}   {cm[0][1] if cm.shape[1] > 1 else 0}")
    print(f"Actual CHD     {cm[1][0]:<12}   {cm[1][1] if cm.shape[1] > 1 else 0}")

    if y_prob is not None and len(np.unique(y_true)) > 1:
        print(f"ROC-AUC   : {roc_auc_score(y_true, y_prob):.4f}")

# ==============================================================================
# ENHANCED TUNING PIPELINE
# ==============================================================================
def run_tuned_pipeline():
    DATASET_DIR = "./dataset_split"
    K_WORDS = 50        # Increased visual vocabulary size from 25 to 50
    NUM_QUBITS = 8      # Quantum feature dimensions

    print("[*] Step 1: Extracting SIFT features...")
    train_map, y_train, train_desc = load_split_descriptors(os.path.join(DATASET_DIR, "train"))
    test_map, y_test, _ = load_split_descriptors(os.path.join(DATASET_DIR, "test"))

    stacked_train = np.vstack([d for d in train_desc if d is not None])
    print(f"[*] Step 2: Fitting K-Means Visual Vocabulary (K = {K_WORDS})...")
    kmeans = KMeans(n_clusters=K_WORDS, random_state=42, n_init=15).fit(stacked_train)

    X_train = build_bovw_histograms(train_map, kmeans, K_WORDS)
    X_test = build_bovw_histograms(test_map, kmeans, K_WORDS)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --------------------------------------------------------------------------
    # 1. Expanded Hyperparameter Tuning for Classical SVM
    # --------------------------------------------------------------------------
    print("\n[*] Step 3: Comprehensive GridSearchCV for Classical SVM...")
    param_grid = {
        'C': [0.1, 1, 10, 100, 1000],
        'gamma': ['scale', 'auto', 0.1, 0.01, 0.001],
        'kernel': ['rbf', 'poly', 'sigmoid']
    }

    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid_search = GridSearchCV(
        SVC(class_weight='balanced', probability=True, random_state=42),
        param_grid,
        cv=cv_strategy,
        scoring='f1',
        n_jobs=-1
    ).fit(X_train_scaled, y_train)

    best_svm = grid_search.best_estimator_
    print(f" -> Optimal Hyperparameters Found: {grid_search.best_params_}")

    y_pred_svm = best_svm.predict(X_test_scaled)
    y_prob_svm = best_svm.predict_proba(X_test_scaled)[:, 1]
    evaluate_performance(y_test, y_pred_svm, y_prob_svm, model_name="Optimized Classical SVM")

    # --------------------------------------------------------------------------
    # 2. Enhanced Quantum SVM (QSVC)
    # --------------------------------------------------------------------------
    print("\n[*] Step 4: Training Enhanced Quantum SVM (QSVC)...")
    pca = PCA(n_components=NUM_QUBITS, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    q_scaler = MinMaxScaler(feature_range=(0, np.pi))
    X_train_q = q_scaler.fit_transform(X_train_pca)
    X_test_q = q_scaler.transform(X_test_pca)

    f_map = zz_feature_map(feature_dimension=NUM_QUBITS, reps=3)  # Reps increased to 3 for entanglement
    qkernel = FidelityQuantumKernel(feature_map=f_map)

    # Increased training sample size to 120 balanced images (60 Non-CHD / 60 CHD)
    idx_non_chd = np.where(y_train == 0)[0][:60]
    idx_chd = np.where(y_train == 1)[0][:60]
    balanced_idx = np.hstack([idx_non_chd, idx_chd])

    qsvc = QSVC(quantum_kernel=qkernel)
    qsvc.fit(X_train_q[balanced_idx], y_train[balanced_idx])

    y_pred_qsvm = qsvc.predict(X_test_q)
    evaluate_performance(y_test, y_pred_qsvm, model_name="Enhanced Quantum SVM")

    return {
        'kmeans': kmeans, 'scaler': scaler, 'svm': best_svm, 'qsvc': qsvc,
        'pca': pca, 'q_scaler': q_scaler, 'X_test_scaled': X_test_scaled,
        'y_test': y_test, 'y_pred_svm': y_pred_svm, 'y_pred_qsvm': y_pred_qsvm
    }

globals_pipeline = run_tuned_pipeline()

[*] Step 1: Extracting SIFT features...
[*] Step 2: Fitting K-Means Visual Vocabulary (K = 50)...

[*] Step 3: Comprehensive GridSearchCV for Classical SVM...
 -> Optimal Hyperparameters Found: {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}

      OPTIMIZED CLASSICAL SVM EVALUATION      
Accuracy  : 60.00%
Precision : 61.11%
Recall    : 55.00%
F1-Score  : 57.89%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
Actual Non_CHD 13             7
Actual CHD     9              11
ROC-AUC   : 0.3875

[*] Step 4: Training Enhanced Quantum SVM (QSVC)...

      ENHANCED QUANTUM SVM EVALUATION      
Accuracy  : 47.50%
Precision : 47.06%
Recall    : 40.00%
F1-Score  : 43.24%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
Actual Non_CHD 11             9
Actual CHD     12             8
CPU times: user 43min 9s, sys: 37.8 s, total: 43min 47s
Wall time: 43min 15s


In [13]:
%%time
import os
import glob
import cv2
import numpy as np
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
)
from skimage.segmentation import chan_vese
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

sift = cv2.SIFT_create(nfeatures=200)

def preprocess_and_segment_4chamber(img_path_or_array, max_iter=100):
    img = cv2.imread(img_path_or_array, cv2.IMREAD_GRAYSCALE) if isinstance(img_path_or_array, str) else img_path_or_array
    if img is None:
        return None, None

    denoised = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    enhanced = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(denoised)

    h, w = enhanced.shape
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(roi_mask, (int(w / 2), int(h / 2)), (int(w * 0.40), int(h * 0.40)), 0, 0, 360, 255, -1)
    focused_img = cv2.bitwise_and(enhanced, enhanced, mask=roi_mask)

    cv_mask = chan_vese(focused_img / 255.0, mu=0.7, lambda1=1.0, lambda2=1.0, tol=1e-3, max_num_iter=max_iter)
    segmented_binary = (cv_mask * 255).astype(np.uint8)

    contours, _ = cv2.findContours(segmented_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boundary_overlay = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(boundary_overlay, contours, -1, (0, 255, 0), 2)

    return segmented_binary, boundary_overlay


def load_split_descriptors(split_dir):
    categories = ['Non_CHD', 'CHD']
    image_desc_map, labels, all_descriptors = [], [], []
    existing_folders = os.listdir(split_dir) if os.path.exists(split_dir) else []

    for label_idx, cat in enumerate(categories):
        target_folder = next((f for f in existing_folders if f.lower() == cat.lower()), None)
        if not target_folder:
            continue

        cat_path = os.path.join(split_dir, target_folder)
        image_files = []
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.BMP', '*.bmp']:
            image_files.extend(glob.glob(os.path.join(cat_path, ext)))

        for img_path in image_files:
            seg_img, _ = preprocess_and_segment_4chamber(img_path, max_iter=30)
            if seg_img is not None:
                _, desc = sift.detectAndCompute(seg_img, None)
                image_desc_map.append(desc)
                labels.append(label_idx)
                if desc is not None:
                    all_descriptors.append(desc)

    return image_desc_map, np.array(labels), all_descriptors


def build_bovw_histograms(image_desc_map, kmeans_model, num_clusters):
    X_bovw = []
    for desc in image_desc_map:
        histogram = np.zeros(num_clusters)
        if desc is not None and len(desc) > 0:
            words = kmeans_model.predict(desc)
            for word in words:
                histogram[word] += 1
            norm = np.linalg.norm(histogram)
            if norm > 0:
                histogram /= norm
        X_bovw.append(histogram)
    return np.array(X_bovw)


def evaluate_performance(y_true, y_pred, y_prob=None, model_name="Model"):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

    print(f"\n==========================================")
    print(f"      {model_name.upper()} EVALUATION      ")
    print(f"==========================================")
    print(f"Accuracy  : {acc * 100:.2f}%")
    print(f"Precision : {prec * 100:.2f}%")
    print(f"Recall    : {rec * 100:.2f}%")
    print(f"F1-Score  : {f1 * 100:.2f}%")

    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:")
    print("               Pred Non_CHD  Pred CHD")
    print(f"Actual Non_CHD {cm[0][0]:<12}   {cm[0][1] if cm.shape[1] > 1 else 0}")
    print(f"Actual CHD     {cm[1][0]:<12}   {cm[1][1] if cm.shape[1] > 1 else 0}")

    if y_prob is not None and len(np.unique(y_true)) > 1:
        print(f"ROC-AUC   : {roc_auc_score(y_true, y_prob):.4f}")


def run_pipeline():
    DATASET_DIR = "./dataset_split"
    K_WORDS, NUM_QUBITS = 25, 8

    print("[*] Extracting SIFT features...")
    train_map, y_train, train_desc = load_split_descriptors(os.path.join(DATASET_DIR, "train"))
    test_map, y_test, _ = load_split_descriptors(os.path.join(DATASET_DIR, "test"))

    stacked_train = np.vstack([d for d in train_desc if d is not None])
    print(f"[*] Training K-Means visual vocabulary (K = {K_WORDS})...")
    kmeans = KMeans(n_clusters=K_WORDS, random_state=42, n_init=10).fit(stacked_train)

    X_train = build_bovw_histograms(train_map, kmeans, K_WORDS)
    X_test = build_bovw_histograms(test_map, kmeans, K_WORDS)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Classical SVM
    print("\n[*] Tuning Classical SVM via GridSearchCV...")
    grid_search = GridSearchCV(
        SVC(class_weight='balanced', probability=True, random_state=42),
        {'C': [0.1, 1, 10], 'gamma': ['scale', 0.1, 0.01], 'kernel': ['rbf', 'linear']},
        cv=5, n_jobs=-1
    ).fit(X_train_scaled, y_train)

    best_svm = grid_search.best_estimator_
    y_pred_svm = best_svm.predict(X_test_scaled)
    y_prob_svm = best_svm.predict_proba(X_test_scaled)[:, 1]
    evaluate_performance(y_test, y_pred_svm, y_prob_svm, model_name="Tuned Classical SVM")

    # Quantum SVM
    print("\n[*] Training Phase-Scaled Quantum SVM (QSVC)...")
    pca = PCA(n_components=NUM_QUBITS, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    q_scaler = MinMaxScaler(feature_range=(0, np.pi))
    X_train_q = q_scaler.fit_transform(X_train_pca)
    X_test_q = q_scaler.transform(X_test_pca)

    f_map = zz_feature_map(feature_dimension=NUM_QUBITS, reps=2)
    qkernel = FidelityQuantumKernel(feature_map=f_map)

    idx_non_chd = np.where(y_train == 0)[0][:30]
    idx_chd = np.where(y_train == 1)[0][:30]
    balanced_idx = np.hstack([idx_non_chd, idx_chd])

    qsvc = QSVC(quantum_kernel=qkernel)
    qsvc.fit(X_train_q[balanced_idx], y_train[balanced_idx])

    y_pred_qsvm = qsvc.predict(X_test_q)
    evaluate_performance(y_test, y_pred_qsvm, model_name="Phase-Scaled Quantum SVM")

    return {
        'kmeans': kmeans, 'scaler': scaler, 'svm': best_svm, 'qsvc': qsvc,
        'pca': pca, 'q_scaler': q_scaler, 'X_test_scaled': X_test_scaled,
        'y_test': y_test, 'y_pred_svm': y_pred_svm, 'y_pred_qsvm': y_pred_qsvm
    }

globals_pipeline = run_pipeline()

[*] Extracting SIFT features...
[*] Training K-Means visual vocabulary (K = 25)...

[*] Tuning Classical SVM via GridSearchCV...

      TUNED CLASSICAL SVM EVALUATION      
Accuracy  : 52.50%
Precision : 52.17%
Recall    : 60.00%
F1-Score  : 55.81%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
Actual Non_CHD 9              11
Actual CHD     8              12
ROC-AUC   : 0.3625

[*] Training Phase-Scaled Quantum SVM (QSVC)...

      PHASE-SCALED QUANTUM SVM EVALUATION      
Accuracy  : 52.50%
Precision : 52.00%
Recall    : 65.00%
F1-Score  : 57.78%

Confusion Matrix:
               Pred Non_CHD  Pred CHD
Actual Non_CHD 8              12
Actual CHD     7              13
CPU times: user 29min 39s, sys: 1min 19s, total: 30min 59s
Wall time: 31min 1s


In [ ]:
%%time
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif'})
os.makedirs('./paper_proofs', exist_ok=True)

y_test = globals_pipeline['y_test']
y_pred_svm = globals_pipeline['y_pred_svm']
y_pred_qsvm = globals_pipeline['y_pred_qsvm']

# 1. Generate Confusion Matrix Figures
print("[*] Generating Confusion Matrices Proof...")
cm_svm = confusion_matrix(y_test, y_pred_svm)
cm_qsvm = confusion_matrix(y_test, y_pred_qsvm)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
labels = ['Non_CHD', 'CHD']
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=labels, yticklabels=labels)
axes[0].set_title("Classical SVM (Test Set)", fontweight='bold')
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(cm_qsvm, annot=True, fmt='d', cmap='Oranges', ax=axes[1], xticklabels=labels, yticklabels=labels)
axes[1].set_title("Quantum SVM Benchmark (Test Set)", fontweight='bold')
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.savefig('./paper_proofs/fig3_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Export Tabular Results
print("[*] Exporting Quantitative Results Table (Table 1)...")
results_data = {
    'Model Architecture': ['Classical Support Vector Machine (RBF)', 'Quantum Support Vector Machine (QSVC)'],
    'Feature Vector': ['25-dim BoVW Histogram', '8 Qubit PCA Features'],
    'Accuracy (%)': [accuracy_score(y_test, y_pred_svm) * 100, accuracy_score(y_test, y_pred_qsvm) * 100],
    'Precision (%)': [precision_score(y_test, y_pred_svm, pos_label=1) * 100, precision_score(y_test, y_pred_qsvm, pos_label=1, zero_division=0) * 100],
    'Recall (%)': [recall_score(y_test, y_pred_svm, pos_label=1) * 100, recall_score(y_test, y_pred_qsvm, pos_label=1, zero_division=0) * 100],
    'F1-Score (%)': [f1_score(y_test, y_pred_svm, pos_label=1) * 100, f1_score(y_test, y_pred_qsvm, pos_label=1, zero_division=0) * 100]
}

df_results = pd.DataFrame(results_data)
df_results.to_csv('./paper_proofs/table1_model_performance.csv', index=False)

print("\n[+] Table 1 Saved to CSV:\n")
print(df_results.to_string(index=False))
print("\n[+] All proof artifacts saved under directory: ./paper_proofs")